In [47]:
import os
import pandas as pd

# Define paths
root_path = "../package_metadata/openml"

# List of folders to process
target_folders = ["kernels_iid_comparison", "other_baselines"]

dataframes_list = []

# Check if root path exists
if os.path.exists(root_path):
    # Iterate through all dataset folders
    for dataset_name in os.listdir(root_path):
        dataset_full_path = os.path.join(root_path, dataset_name)
        
        # Ensure it is a directory
        if os.path.isdir(dataset_full_path):
            
            # Iterate through the specified target folders
            for folder_name in target_folders:
                target_path = os.path.join(dataset_full_path, folder_name)
                
                # Check if the target folder exists in this dataset
                if os.path.exists(target_path):
                    # Iterate through all files inside
                    for filename in os.listdir(target_path):
                        if filename.endswith(".csv"):
                            file_path = os.path.join(target_path, filename)
                            
                            try:
                                # Read the CSV file
                                df = pd.read_csv(file_path)
                                
                                # --- 1. EXTRACT MODEL NAME FROM FILENAME ---
                                if "_ann" in filename:
                                    df["model_name"] = "nn"
                                elif "_xgboost" in filename:
                                    df["model_name"] = "xgboost"
                                else:
                                    df["model_name"] = "unknown"
                                
                                # --- 2. EXISTING COLUMNS LOGIC ---
                                if "data_modification_method" not in df.columns:
                                    df["data_modification_method"] = "none"
                                
                                df["dataset_name"] = dataset_name
                                
                                # The compression_coeff logic has been removed as requested
                                
                                dataframes_list.append(df)
                                
                            except Exception as e:
                                print(f"Error reading file {file_path}: {e}")

if dataframes_list:
    df_results = pd.concat(dataframes_list, ignore_index=True)
    print(f"Loaded {len(df_results)} rows total.")
else:
    print("No data found.")

Error reading file ../package_metadata/openml/numerai28.6/other_baselines/kernel_thinning_stratified_sage_permutation_ann_42_10.csv: No columns to parse from file
Loaded 77605 rows total.


In [48]:
df_filtered = df_results[
    (df_results["data_modification_method"] == "stratified") |
    (df_results["data_modification_method"] == "predictions") |
    ((df_results["method"] == "kernel_thinning") & (df_results['kernel'] == "gaussian")) |
    (df_results["method"] == "iid") |
    (df_results["method"] == "arfpy") |
    (df_results["method"] == "stein_thinning")
]

In [49]:
df_filtered = df_filtered.reset_index(drop=True)

mapping = {
    "none": "kt",
    "stratified": "kt_stratified",
    "predictions": "kt_predictions"
}
df_filtered['method_total'] = df_filtered['data_modification_method'].map(mapping)

df_filtered.loc[df_filtered['method'] == 'iid', 'method_total'] = 'iid'
df_filtered.loc[df_filtered['method'] == 'arfpy', 'method_total'] = 'arfpy'
df_filtered.loc[df_filtered['method'] == 'stein_thinning', 'method_total'] = 'stein_thinning'

In [50]:
def get_explainer_total(row):
    ex = row['explainer']
    st = row['strategy']
    
    if ex == 'expected_gradients':
        return 'expected_gradients'

    elif ex == 'shap' and st == 'kernel':
        return 'shap_kernel'
    
    elif ex == 'shapiq' and st == 'kernel':
        return 'shapiq_kernel'
        
    elif ex == 'sage' and st == 'permutation':
        return 'sage_permutation'

df_filtered['explainer_total'] = df_filtered.apply(get_explainer_total, axis=1)

In [51]:
cols_to_drop = [
    "explanation_time", 
    "explainer", 
    "strategy", 
    "method", 
    "g", 
    "num_bins", 
    "target_size", 
    "unique_samples", 
    "data_modification_method"
]
df_filtered = df_filtered.drop(columns=cols_to_drop, errors='ignore')
df_filtered

,mae,top_k,size,mmd,kernel,compression_time,model_name,dataset_name,method_total,explainer_total
0,0.015681,0.928516,16,0.016523,gaussian,0.514049,nn,nomao,kt,expected_gradients
1,0.010362,0.932178,32,0.006612,gaussian,0.451655,nn,nomao,kt,expected_gradients
2,0.007373,0.964844,64,0.003079,gaussian,0.495402,nn,nomao,kt,expected_gradients
3,0.004830,0.975391,128,0.001234,gaussian,0.489754,nn,nomao,kt,expected_gradients
4,0.004315,0.972998,256,0.000666,gaussian,0.491907,nn,nomao,kt,expected_gradients
...,...,...,...,...,...,...,...,...,...,...
50375,0.044513,0.738623,16,0.044701,Not needed,62.681515,nn,california_housing,arfpy,shapiq_kernel
50376,0.028856,0.802686,32,0.023767,Not needed,61.762486,nn,california_housing,arfpy,shapiq_kernel
50377,0.024603,0.846191,64,0.009843,Not needed,62.577947,nn,california_housing,arfpy,shapiq_kernel
50378,0.015544,0.902002,128,0.006721,Not needed,62.833208,nn,california_housing,arfpy,shapiq_kernel


In [52]:
selected_dataset = "nomao"

df_one_dataset = df_filtered[df_filtered["dataset_name"] == selected_dataset].reset_index(drop=True)

df_one_dataset

,mae,top_k,size,mmd,kernel,compression_time,model_name,dataset_name,method_total,explainer_total
0,0.015681,0.928516,16,0.016523,gaussian,0.514049,nn,nomao,kt,expected_gradients
1,0.010362,0.932178,32,0.006612,gaussian,0.451655,nn,nomao,kt,expected_gradients
2,0.007373,0.964844,64,0.003079,gaussian,0.495402,nn,nomao,kt,expected_gradients
3,0.004830,0.975391,128,0.001234,gaussian,0.489754,nn,nomao,kt,expected_gradients
4,0.004315,0.972998,256,0.000666,gaussian,0.491907,nn,nomao,kt,expected_gradients
...,...,...,...,...,...,...,...,...,...,...
255,0.094507,0.406445,16,0.163549,Not needed,0.572714,nn,nomao,stein_thinning,expected_gradients
256,0.070435,0.578223,32,0.129386,Not needed,2.029501,nn,nomao,stein_thinning,expected_gradients
257,0.058105,0.669531,64,0.097028,Not needed,1.246105,nn,nomao,stein_thinning,expected_gradients
258,0.052621,0.693359,128,0.077313,Not needed,3.105820,nn,nomao,stein_thinning,expected_gradients


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

unique_datasets = df_filtered["dataset_name"].unique()

all_methods = sorted(df_filtered['method_total'].unique())
palette = sns.color_palette("tab10", n_colors=len(all_methods))
method_colors = dict(zip(all_methods, palette))

models = ['nn', 'xgboost']

for dataset_name in unique_datasets:
    print(f"Processing dataset: {dataset_name}...")
    
    output_folder = f"Image3/{dataset_name}"
    os.makedirs(output_folder, exist_ok=True)
    
    df_one_dataset = df_filtered[df_filtered["dataset_name"] == dataset_name].copy()
    
    explainers = df_one_dataset['explainer_total'].unique()
    n_cols = len(explainers)
    
    if n_cols == 0:
        continue

    fig, axes = plt.subplots(
        nrows=2, 
        ncols=n_cols, 
        figsize=(7 * n_cols, 12), 
        squeeze=False 
    )
    
    fig.suptitle(f"Dataset: {dataset_name} - MAE vs Size", fontsize=20, y=1.02)

    for col_idx, explainer in enumerate(explainers):
        for row_idx, model in enumerate(models):
            ax = axes[row_idx, col_idx]
            
            subset = df_one_dataset[
                (df_one_dataset['explainer_total'] == explainer) & 
                (df_one_dataset['model_name'] == model)
            ]
            
            if not subset.empty:
                agg_data = subset.groupby(['size', 'method_total'])['mae'].agg(['mean', 'std']).reset_index()
                
                for method in all_methods:
                    method_data = agg_data[agg_data['method_total'] == method]
                    
                    if not method_data.empty:
                        ax.errorbar(
                            x=method_data['size'], 
                            y=method_data['mean'], 
                            yerr=method_data['std'], 
                            label=method,
                            color=method_colors[method],
                            fmt='-o',    
                            capsize=5,
                            elinewidth=2,
                            capthick=2, 
                            alpha=0.8
                        )
                
                ax.set_title(f"Model: {model} | Explainer: {explainer}", fontsize=14)
                ax.set_xlabel("Subset Size")
                ax.set_xscale('log')
                ax.set_ylabel("MAE")
                ax.grid(True, linestyle='--', alpha=0.5)
                
                ax.legend(title='Method', loc='upper right', fontsize='small')
            else:
                ax.set_visible(False)

    plt.tight_layout()
    
    filename = f"{output_folder}/mae_comparison_{dataset_name}.png"
    plt.savefig(filename, bbox_inches='tight')
    
    plt.close()

print("All plots generated and saved.")

Processing dataset: nomao...
Processing dataset: wilt...
Processing dataset: isolet...
Processing dataset: white_wine...
Processing dataset: wave_energy...
Processing dataset: pumadyn32nh...
Processing dataset: pendigits...
Processing dataset: kings_county...
Processing dataset: brazilian_houses...
Processing dataset: abalone...
Processing dataset: sarcos...
Processing dataset: physiochemical_protein...
Processing dataset: jungle_chess_2pcs_raw_endgame_complete...
Processing dataset: kin8nm...
Processing dataset: miami_housing...
Processing dataset: diamonds...
Processing dataset: letter...
Processing dataset: naval_propulsion_plant...
Processing dataset: PhishingWebsites...
Processing dataset: first-order-theorem-proving...
Processing dataset: grid_stability...
Processing dataset: phoneme...
Processing dataset: spambase...
Processing dataset: superconductivity...
Processing dataset: optdigits...
Processing dataset: bank-marketing...
Processing dataset: satimage...
Processing dataset: 